# EIA-860 renewable-site regridding

This notebook filters the 2024 operable MISO fleet to onshore wind and solar PV, combines generators at shared site coordinates, and matches those sites to weather-grid IDs. Production mappings are prepared separately; this example does not reconstruct every BA or the sup3rCC grids.

**Inputs:** The 2024 EIA-860 plant and operable-generator workbooks, live WTK/BC-HRRR/NSRDB grid metadata, and a separately supplied MISO subregion mapping for the appendix.

**Requirements:** The repository environment and a local HSDS service at `http://localhost:5101` with access to `nrel-pds-hsds` for Step 5. Workbook preparation and the appendix use bundled files.

**Outputs:** A combined MISO example CSV under `../../data_outputs/data_flow/eia860_regridding_methodology/`, plus capacity and mapping summaries. The [site-CF notebook](site_cf_generation_ba_weighting_validation.ipynb) continues the method using its own **2022** MISO fleet and 2023 weather, separate from this example's output.

Run the code cells from top to bottom. See the [setup instructions](../../README.md#quick-start).


**Saved-output provenance:** Workbook preparation, site aggregation and the supplied subregion summary were checked in fresh kernels during an earlier review. The grid-matching results are retained from an earlier successful live HSDS run; a local-only run does not refresh those lookups.


## Setup

In [1]:
from pathlib import Path

from IPython.display import display
import pandas as pd
from rex import Resource
from scipy.spatial import cKDTree

PLANT_XLSX = Path("../../data_inputs/eia860/2024_raw/2___Plant_Y2024.xlsx")
GENERATOR_XLSX = Path("../../data_inputs/eia860/2024_raw/3_1_Generator_Y2024.xlsx")
OUTPUT_DIR = Path("../../data_outputs/data_flow/eia860_regridding_methodology")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: Read EIA-860 workbooks

The plant workbook supplies location and BA fields. The `Operable` generator sheet supplies technology, generator identifiers and nameplate capacity. The preview below identifies the two workbook sheets and their dimensions. Subsequent previews retain the EIA source headers.


In [2]:
plant = pd.read_excel(PLANT_XLSX, sheet_name="Plant", header=1)
generator = pd.read_excel(GENERATOR_XLSX, sheet_name="Operable", header=1)

workbook_summary = pd.DataFrame(
    [
        {
            "table": "plant",
            "workbook": PLANT_XLSX.name,
            "sheet": "Plant",
            "rows": len(plant),
            "columns": len(plant.columns),
        },
        {
            "table": "generator",
            "workbook": GENERATOR_XLSX.name,
            "sheet": "Operable",
            "rows": len(generator),
            "columns": len(generator.columns),
        },
    ]
)
workbook_summary = workbook_summary.rename(
    columns={
        "table": "Table",
        "workbook": "Workbook",
        "sheet": "Sheet",
        "rows": "Rows",
        "columns": "Columns",
    }
)
display(
    workbook_summary.style.hide(axis="index").format(
        {
            "Rows": "{:,.0f}",
            "Columns": "{:,.0f}",
        }
    )
)

Table,Workbook,Sheet,Rows,Columns
plant,2___Plant_Y2024.xlsx,Plant,"16,132",42
generator,3_1_Generator_Y2024.xlsx,Operable,"26,856",73


## Step 2: Add plant fields to generator rows

Merge plant coordinates and balancing-authority fields onto each generator. Keep the five fields needed for filtering and aggregation, plus plant code, plant name, and generator ID for readable examples.

In [3]:
MERGE_KEYS = [
    "Utility ID",
    "Utility Name",
    "Plant Code",
    "Plant Name",
    "State",
    "County",
    "Sector",
    "Sector Name",
]

MERGED_COLUMNS_TO_KEEP = [
    "Plant Code",
    "Plant Name",
    "Generator ID",
    "Technology",
    "Nameplate Capacity (MW)",
    "Latitude",
    "Longitude",
    "Balancing Authority Code",
]

# Attach plant fields to each generator row.
generator_fields = MERGE_KEYS + ["Generator ID", "Technology", "Nameplate Capacity (MW)"]
plant_fields = MERGE_KEYS + ["Latitude", "Longitude", "Balancing Authority Code"]
generator_to_merge = generator[generator_fields]
plant_to_merge = plant[plant_fields]
merged_raw = generator_to_merge.merge(plant_to_merge, on=MERGE_KEYS, how="left", indicator=True)
merged = merged_raw[MERGED_COLUMNS_TO_KEEP].copy()

# Clean the columns used in later filters and capacity sums.
ba_codes = merged["Balancing Authority Code"].astype(str)
ba_codes = ba_codes.str.strip()
merged["Balancing Authority Code"] = ba_codes.str.upper()
merged["Nameplate Capacity (MW)"] = pd.to_numeric(merged["Nameplate Capacity (MW)"])
merged["Latitude"] = pd.to_numeric(merged["Latitude"])
merged["Longitude"] = pd.to_numeric(merged["Longitude"])

merge_summary = pd.DataFrame(
    [
        {"check": "generator rows", "value": len(generator)},
        {"check": "merged rows", "value": len(merged_raw)},
        {"check": "unmatched generator rows", "value": int((merged_raw["_merge"] != "both").sum())},
        {
            "check": "merged rows missing lat/lon",
            "value": int(merged[["Latitude", "Longitude"]].isna().any(axis=1).sum()),
        },
    ]
)

display(merge_summary.rename(columns={"check": "Check", "value": "Rows"}).style.hide(axis="index").format({"Rows": "{:,.0f}"}))

display(merged.head(5))

Check,Rows
generator rows,"26,856"
merged rows,"26,856"
unmatched generator rows,2
merged rows missing lat/lon,2


,Plant Code,Plant Name,Generator ID,Technology,Nameplate Capacity (MW),Latitude,Longitude,Balancing Authority Code
0,1.0,Sand Point,1,Petroleum Liquids,0.9,55.339722,-160.497222,NAN
1,1.0,Sand Point,2,Petroleum Liquids,0.9,55.339722,-160.497222,NAN
2,1.0,Sand Point,3,Petroleum Liquids,0.5,55.339722,-160.497222,NAN
3,1.0,Sand Point,5.1,Petroleum Liquids,0.4,55.339722,-160.497222,NAN
4,1.0,Sand Point,WT1,Onshore Wind Turbine,0.5,55.339722,-160.497222,NAN


## Step 3: Filter to wind/solar for one BA

This step starts with the merged EIA-860 generator table, keeps only wind and solar generator rows, and then keeps only rows assigned to MISO. 

The filtered table, `ba_renewable`, is used in Step 4.


In [4]:
RENEWABLE_TECHNOLOGIES = ["Onshore Wind Turbine", "Solar Photovoltaic"]

# Keep only the renewable generator technologies used by the wind/solar CF workflow.
renewable = merged[merged["Technology"].isin(RENEWABLE_TECHNOLOGIES)].copy()

# Keep only renewable generator rows assigned to the selected BA.
ba_renewable = renewable[renewable["Balancing Authority Code"].eq("MISO")].copy()

filter_summary = pd.DataFrame(
    [
        {"check": "all generator rows", "value": len(merged)},
        {"check": "renewable generator rows", "value": len(renewable)},
        {"check": "MISO renewable generator rows", "value": len(ba_renewable)},
        {
            "check": "MISO renewable rows missing lat/lon",
            "value": int(ba_renewable[["Latitude", "Longitude"]].isna().any(axis=1).sum()),
        },
    ]
)

display(filter_summary.rename(columns={"check": "Check", "value": "Rows"}).style.hide(axis="index").format({"Rows": "{:,.0f}"}))

Check,Rows
all generator rows,"26,856"
renewable generator rows,"8,702"
MISO renewable generator rows,"1,579"
MISO renewable rows missing lat/lon,0


## Step 4: Aggregate generator rows to site coordinates

This step combines MISO generators sharing `Technology`, `Latitude` and `Longitude` into one site row, summing their nameplate capacity.

The first cell shows one real example: multiple generator rows at the same technology and plant coordinates before aggregation, followed by the single site row created after their `Nameplate Capacity (MW)` values are summed.

The second cell applies the same aggregation to all rows in `ba_renewable`. Wind and solar remain separate because `Technology` is included in the grouping.

In [5]:
SITE_GROUP_COLUMNS = ["Technology", "Latitude", "Longitude"]

with_coordinates = ba_renewable.dropna(subset=["Latitude", "Longitude"])

site_groups = with_coordinates.groupby(SITE_GROUP_COLUMNS)
site_row_counts = site_groups.size()
site_row_counts = site_row_counts.reset_index(name="generator_rows")
duplicate_site = site_row_counts[site_row_counts["generator_rows"] > 1].iloc[0]

same_site = ((with_coordinates["Technology"] == duplicate_site["Technology"]) & (with_coordinates["Latitude"] == duplicate_site["Latitude"]) & (with_coordinates["Longitude"] == duplicate_site["Longitude"]))

before = with_coordinates[same_site]
print("Before aggregation: generator rows at the same technology and lat/lon")
display(before[MERGED_COLUMNS_TO_KEEP])

after = before.groupby(SITE_GROUP_COLUMNS, as_index=False).agg(site_nameplate_mw=("Nameplate Capacity (MW)", "sum"))
print("After aggregation: one site row with summed nameplate capacity")
display(after.rename(columns={"site_nameplate_mw": "Site nameplate capacity (MW)"}).style.hide(axis="index").format({"Latitude": "{:.6f}", "Longitude": "{:.6f}", "Site nameplate capacity (MW)": "{:,.1f}"}))

Before aggregation: generator rows at the same technology and lat/lon


,Plant Code,Plant Name,Generator ID,Technology,Nameplate Capacity (MW),Latitude,Longitude,Balancing Authority Code
17748,59637.0,Adams Wind,ADWF,Onshore Wind Turbine,4.7,40.92,-94.671667,MISO
17749,59637.0,Adams Wind,ADWF2,Onshore Wind Turbine,43.3,40.92,-94.671667,MISO
17750,59637.0,Adams Wind,ADWF3,Onshore Wind Turbine,58.0,40.92,-94.671667,MISO
17751,59637.0,Adams Wind,ADWF4,Onshore Wind Turbine,48.3,40.92,-94.671667,MISO


After aggregation: one site row with summed nameplate capacity


Technology,Latitude,Longitude,Site nameplate capacity (MW)
Onshore Wind Turbine,40.920000,-94.671667,154.3


In [6]:
ba_site_capacity = with_coordinates.groupby(SITE_GROUP_COLUMNS, as_index=False).agg(site_nameplate_mw=("Nameplate Capacity (MW)", "sum"))

generator_summary = ba_renewable.groupby("Technology", as_index=False).agg(generator_rows=("Technology", "size"))

site_summary = ba_site_capacity.groupby("Technology", as_index=False).agg(unique_site_coordinates=("site_nameplate_mw", "size"), site_nameplate_mw=("site_nameplate_mw", "sum"))

ba_site_summary = generator_summary.merge(site_summary, on="Technology")

wind_sites = ba_site_capacity["Technology"].eq("Onshore Wind Turbine")
solar_sites = ba_site_capacity["Technology"].eq("Solar Photovoltaic")
wind_site_capacity = ba_site_capacity[wind_sites].copy()
solar_site_capacity = ba_site_capacity[solar_sites].copy()

print("MISO renewable generator rows aggregated to site coordinates")
display(
    ba_site_summary.rename(
        columns={
            "generator_rows": "Generator rows",
            "unique_site_coordinates": "Sites",
            "site_nameplate_mw": "Nameplate capacity (MW)",
        }
    )
    .style.hide(axis="index")
    .format({"Generator rows": "{:,.0f}", "Sites": "{:,.0f}", "Nameplate capacity (MW)": "{:,.1f}"})
)

MISO renewable generator rows aggregated to site coordinates


Technology,Generator rows,Sites,Nameplate capacity (MW)
Onshore Wind Turbine,427,356,"32,150.7"
Solar Photovoltaic,"1,152",954,"13,574.9"


## Step 5: Match BA site coordinates to HSDS grid points

This step maps renewable site coordinates to the WTK, BC-HRRR, and NSRDB grid IDs used by the site-weather workflow. Rows sharing a selected gid are grouped and their EIA-860 nameplate capacity is summed.

Before running this step, start the local HSDS service at `http://localhost:5101` with access to the `nrel-pds-hsds` bucket. The reconstruction reads live grid coordinates and summarizes the resulting grid-point counts and nameplate totals; it does not compare against a packaged site CSV.

In [7]:
HSDS_ENDPOINT = "http://localhost:5101"
HSDS_API_KEY = None
HSDS_BUCKET = "nrel-pds-hsds"

# Use one representative resource year for each source's historical grid.
RESOURCE_PATHS = {
    "wtk": "/nrel/wtk/conus/wtk_conus_2013.h5",
    "bchrrr": "/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5",
    "nsrdb": "/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5",
}
BA_REGRIDDED_POINTS_CSV = OUTPUT_DIR / "MISO_2024_regridded_hsds_grid_points.csv"

site_capacity_by_source = {
    "wtk": wind_site_capacity,
    "bchrrr": wind_site_capacity,
    "nsrdb": solar_site_capacity,
}
regridded_tables = []
for weather_source, resource_path in RESOURCE_PATHS.items():
    # Select the wind or solar sites for this weather source.
    sites = site_capacity_by_source[weather_source]

    # 1. Read the latitude/longitude grid for this weather source.
    print(f"Looking up {weather_source} grid metadata through HSDS: {resource_path}")
    with Resource(
        resource_path,
        hsds=True,
        hsds_kwargs={
            "endpoint": HSDS_ENDPOINT,
            "api_key": HSDS_API_KEY,
            "bucket": HSDS_BUCKET,
        },
    ) as resource:
        grid_coordinates = resource.coordinates

    # 2. The coordinate row number is the HSDS grid ID.
    tree = cKDTree(grid_coordinates)
    _, selected_gids = tree.query(sites[["Latitude", "Longitude"]].to_numpy(dtype=float))
    selected_coordinates = grid_coordinates[selected_gids]

    # 3. Store the selected grid IDs and coordinates for this weather source.
    matched = sites.assign(weather_source=weather_source, selected_gid=selected_gids, grid_lat=selected_coordinates[:, 0], grid_lon=selected_coordinates[:, 1])

    # 4. Sum capacity when multiple sites map to the same gid.
    regridded_tables.append(matched.groupby(["weather_source", "Technology", "selected_gid", "grid_lat", "grid_lon"], as_index=False)["site_nameplate_mw"].sum())

# Combine the results and write the reconstructed grid-point file.
nearest_grid_points = pd.concat(regridded_tables, ignore_index=True)
nearest_grid_points.to_csv(BA_REGRIDDED_POINTS_CSV, index=False)

print(f'Wrote HSDS grid-point reconstruction ({len(nearest_grid_points):,} rows):\n  {BA_REGRIDDED_POINTS_CSV.as_posix()}')

# Summarize the grid-point count and total capacity by weather source.
display(
    nearest_grid_points.groupby("weather_source", as_index=False).agg(grid_points=("selected_gid", "nunique"), nameplate_mw=("site_nameplate_mw", "sum")).rename(
        columns={
            "weather_source": "Weather source",
            "grid_points": "Grid points",
            "nameplate_mw": "Nameplate capacity (MW)",
        }
    )
    .style.hide(axis="index")
    .format(
        {
            "Grid points": "{:,.0f}",
            "Nameplate capacity (MW)": "{:,.1f}",
        }
    )
)

Looking up wtk grid metadata through HSDS: /nrel/wtk/conus/wtk_conus_2013.h5


Looking up bchrrr grid metadata through HSDS: /nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5


Looking up nsrdb grid metadata through HSDS: /nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5


Wrote HSDS grid-point reconstruction (1,355 rows):
  ../../data_outputs/data_flow/eia860_regridding_methodology/MISO_2024_regridded_hsds_grid_points.csv


Weather source,Grid points,Nameplate capacity (MW)
bchrrr,317,"32,150.7"
nsrdb,721,"13,574.9"
wtk,317,"32,150.7"


## Interpretation

The output assigns 2024 MISO wind and solar capacity to weather-grid points. Generators first combine at shared technology and coordinates; those site capacities combine again when they select the same grid ID. Wind and solar remain separate throughout. The following appendix explains the supplied MISO subregion assignments; the separate site-CF validation notebook uses a 2022 fleet.


## Appendix A: MISO renewable subregion mapping

For MISO renewables, the subregion split happens only at the weighted-CF step:

- `MISO_wtk.csv`, `MISO_bchrrr.csv`, and `MISO_nsrdb.csv` are total-MISO regridded reference files.
- Site-weather H5 files are total-MISO files.
- Site-CF H5 files are total-MISO files.
- The spreadsheet `eia860_2024_operable_miso_subregions.xlsx` maps each total-MISO renewable `gid` to `miso_lrz`, `ba_code`, and `ba_number`.
- `ba_code` identifies the MISO renewable subregion each row belongs to, such as `MISO_0001`.
- The weighted BA-level CF step filters the total-MISO site-CF H5 by `ba_code` and writes both total-MISO and MISO subregion CF CSVs.

The code below reads the complete packaged spreadsheet. The displayed summary is limited to the WTK, BC-HRRR, and NSRDB sources used in this example, followed by one BC-HRRR mapping row per MISO renewable subregion. The full mapping and summary retain all weather sources.



In [7]:
MISO_SUBREGION_SPREADSHEET = Path("../../data_inputs/eia860/2024_regridded_points/eia860_2024_operable_miso_subregions.xlsx")

print(f"MISO renewable subregion spreadsheet: {MISO_SUBREGION_SPREADSHEET.as_posix()}")

miso_ba_map = pd.read_excel(MISO_SUBREGION_SPREADSHEET, sheet_name="mapping", dtype=str)
miso_ba_map = miso_ba_map.fillna("")
miso_ba_map["gid"] = pd.to_numeric(miso_ba_map["gid"], errors="coerce")
subregion_keys = ["source_key", "ba_code", "ba_number"]
subregion_groups = miso_ba_map.groupby(subregion_keys, as_index=False)
miso_ba_summary = subregion_groups.agg(rows=("gid", "size"), unique_gids=("gid", "nunique"))
miso_ba_summary = miso_ba_summary.sort_values(subregion_keys)

preview_columns = [
    "generator_set",
    "source_key",
    "gid",
    "lat",
    "lon",
    "miso_lrz",
    "ba_code",
    "ba_number",
    "assignment_method",
]

print("WTK, BC-HRRR, and NSRDB subregion assignment summary")
display(
    miso_ba_summary.loc[miso_ba_summary["source_key"].isin(["wtk", "bchrrr", "nsrdb"])].rename(
        columns={
            "source_key": "Weather source",
            "ba_code": "BA code",
            "ba_number": "BA number",
            "rows": "Rows",
            "unique_gids": "Unique grid IDs",
        }
    )
    .style.hide(axis="index")
    .format({"Rows": "{:,.0f}", "Unique grid IDs": "{:,.0f}"})
)
print("One BC-HRRR mapping example per MISO renewable subregion")
display(miso_ba_map.loc[miso_ba_map["source_key"].eq("bchrrr"), preview_columns].sort_values(["ba_code", "gid"]).drop_duplicates("ba_code"))

MISO renewable subregion spreadsheet: ../../data_inputs/eia860/2024_regridded_points/eia860_2024_operable_miso_subregions.xlsx


WTK, BC-HRRR, and NSRDB subregion assignment summary


Weather source,BA code,BA number,Rows,Unique grid IDs
bchrrr,MISO_0001,0001,133,133
bchrrr,MISO_0004,0004,23,23
bchrrr,MISO_0006,0006,7,7
bchrrr,MISO_0027,0027,42,42
bchrrr,MISO_0035,0035,111,111
bchrrr,MISO_8910,8910,1,1
nsrdb,MISO_0001,0001,320,320
nsrdb,MISO_0004,0004,100,100
nsrdb,MISO_0006,0006,71,71
nsrdb,MISO_0027,0027,135,135


One BC-HRRR mapping example per MISO renewable subregion


,generator_set,source_key,gid,lat,lon,miso_lrz,ba_code,ba_number,assignment_method
0,eia860_2024_operable,bchrrr,936134,46.277283,-104.19415,LRZ1,MISO_0001,0001,shapefile_overlay
245,eia860_2024_operable,bchrrr,1557619,39.61221,-90.837585,LRZ4,MISO_0004,0004,shapefile_overlay
281,eia860_2024_operable,bchrrr,1719814,40.69532,-87.47711,LRZ6,MISO_0006,0006,shapefile_overlay
244,eia860_2024_operable,bchrrr,1541999,43.72555,-90.80496,LRZ2,MISO_0027,0027,shapefile_overlay
81,eia860_2024_operable,bchrrr,1297140,42.842155,-95.936554,LRZ3,MISO_0035,0035,shapefile_overlay
256,eia860_2024_operable,bchrrr,1603681,34.47388,-90.39209,LRZ10,MISO_8910,8910,shapefile_overlay
